# PromptForge Phase 2 — Optimizer (package driver)

Uses `src/promptforge` (same code as local `scripts/train_optimizer.py`).

**Runtime → GPU**


In [ ]:
import torch
assert torch.cuda.is_available(), "Enable GPU runtime"
print(torch.cuda.get_device_name(0))


In [ ]:
import os
from pathlib import Path

REPO_DIR = Path("/content/promptModel")
# Upload/clone repo first, then:
# !git clone https://github.com/YOUR_USER/promptModel.git /content/promptModel
if (Path.cwd() / "src" / "promptforge").exists():
    REPO_DIR = Path.cwd()
os.chdir(REPO_DIR)
print(REPO_DIR.resolve())


In [ ]:
!pip install -q -U pip
!pip install -q -e .
!pip install -q peft

import promptforge
print(promptforge.__version__)


In [ ]:
import os
os.environ["PROMPTFORGE_REQUIRE_GPU"] = "1"

from promptforge.config import OptimizerConfig
from promptforge.training import train_optimizer

config = OptimizerConfig.from_yaml("configs/optimizer.yaml")
config.dataset_path = "/content/promptforge_optimizer_dataset.csv"
config.output_dir = "/content/outputs/promptforge-optimizer"
config.final_model_dir = "/content/outputs/promptforge-optimizer-model"
config.num_examples = 10000
config.num_train_epochs = 2
config.prefer_gpu = True
config.use_fp16 = True

metrics = train_optimizer(config, regenerate_dataset=True)
metrics


In [ ]:
from promptforge import PromptForge

pf = PromptForge(
    optimizer_model_path="/content/outputs/promptforge-optimizer-model",
    prefer_gpu=True,
)

for prompt in ["Make an app.", "Build me a website.", "Make a Python API for beginners."]:
    result = pf.optimize(
        prompt,
        analysis={
            "quality_score": 20,
            "dimensions": {
                "clarity": 30,
                "specificity": 12,
                "context": 10,
                "goal_definition": 25,
                "constraints": 8,
                "completeness": 15,
                "actionability": 18,
            },
            "issues": ["too_vague", "missing_context"],
            "missing_information": ["context", "constraints", "output_format"],
        },
        task_type="coding",
    )
    print("=" * 80)
    print("ORIGINAL:", prompt)
    print("OPTIMIZED:\n", result["optimized_prompt"])
